In [ ]:
# Cấu hình: sửa tại đây, chọn CPU + Internet rồi Run All / Save Version (Run All).
RERANK = False
SEARCH_MODE = "4-branch"
BRANCH_LIMIT = 100
TOP_K = 150

WORKERS = 2
BATCH_SIZE = 8
REQUEST_TIMEOUT = 300
QUERY_RETRIES = 2                # Thử lại lỗi mạng qua checkpoint; không chạy lại query đã đạt
QDRANT_SEARCH_THREADS = 2
RESTORE_TIMEOUT = 3600
PARITY_QUERIES = 3
LIMIT = None                    # None: toàn bộ query; số nguyên: chạy thử số query đó
START = None                    # Vị trí dòng 1-based, inclusive
END = None
MAX_DOCS = None
MAX_CHUNKS = None
QUERY_FILTER = None

QUERY_PATH = None               # Tự tìm đúng tên tệp trong /kaggle/input
SOURCE_PATH = None
SNAPSHOT_MANIFEST_PATH = None
TEMP_ROOT = "/kaggle/temp/sbtc-retrieval"
OUTPUT_DIR = "/kaggle/working/retrieval"
OUTPUT_NAME = "submission"
RESUME = True                   # Tiếp tục checkpoint nếu cell truy xuất bị gián đoạn

RERANKER_BATCH_SIZE = 1
RERANKER_MAX_LENGTH = 8192
RERANKER_TOKEN_BUDGET = 8192


## Truy xuất CPU từ snapshot GTE + BM25

Giữ nguyên `BackEnd.app.pipeline.run_pipeline` và weighted RRF theo `dense_weight`
của từng query. Dùng vector có sẵn trong Parquet, không tải GTE hoặc tạo lại embedding.
SQLite chỉ giữ vị trí byte trong JSONL/JSON; văn bản đầu ra lấy nguyên văn từ bản
sao byte-for-byte trong temp, tạo cùng lượt quét/checksum nguồn để tránh random I/O trên input.
Snapshot, Qdrant, SQLite và cache nằm trong `/kaggle/temp`; chỉ kết quả và báo cáo
được lưu ở `/kaggle/working/retrieval`.

Inputs: dataset `nhminh107/sbtc-datav2-etl-20261016`, output notebook
`nhminh107/sbtc-gte-qdrant-snapshot-2gpu-20261017`, và dataset query đi kèm.
Kiểm tra đối chiếu payload Qdrant ↔ JSON trên vài query trước khi chạy toàn bộ;
kiểm tra ID, schema, văn bản gốc và ZIP sau khi hoàn tất. Nếu lỗi, cell dừng và giữ
checkpoint; không tạo bài nộp hoàn chỉnh từ kết quả thiếu query.


In [ ]:
import atexit
import base64
from copy import deepcopy
import hashlib
import importlib.util
import io
import json
import logging
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
import zipfile

if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Notebook này chỉ chạy/kiểm thử trên Kaggle.")
TEMP = Path(TEMP_ROOT).resolve()
if not TEMP.is_relative_to(Path("/kaggle/temp")):
    raise ValueError("TEMP_ROOT phải nằm dưới /kaggle/temp")
TEMP.mkdir(parents=True, exist_ok=True)
OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
for name, value in {
    "TMPDIR": str(TEMP / "tmp"), "HF_HOME": str(TEMP / "huggingface"),
    "XDG_CACHE_HOME": str(TEMP / "cache"), "PIP_CACHE_DIR": str(TEMP / "pip-cache"),
    "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "TOKENIZERS_PARALLELISM": "false",
}.items():
    os.environ[name] = value
    if name.endswith("HOME") or name in {"TMPDIR", "PIP_CACHE_DIR"}:
        Path(value).mkdir(parents=True, exist_ok=True)
requirements = {"requests": "requests>=2.32,<3", "pyarrow": "pyarrow>=18",
                "sqlalchemy": "sqlalchemy>=2,<3", "dotenv": "python-dotenv>=1,<2",
                "psutil": "psutil>=5"}
missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", *missing], check=True)
BUNDLE_SHA256 = '5edaa46bef901a6fd90e388dedb670e2c962cc93cb1a269d2f1998f43dc2272a'
SOURCE_HASHES = {'BackEnd/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'BackEnd/app/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'BackEnd/app/pipeline.py': 'a2fc9c16f59d7bdfb891f263fdc7b523a3a7355af2cd29f0cb0c5212746724e5', 'BackEnd/app/Database/__init__.py': 'fc87f32f701026503f87111d0de9e690901488961fce9350354e5809ba7ecd0f', 'BackEnd/app/Database/models.py': '88213721de22294f44268fa6a2ebd43a379e0368dc15955f3e357d49d5a0c6a4', 'BackEnd/app/Database/adapter.py': '6860ebdc45643395b296776cff88fb817deea5b1108a4f644fce4a75338ee62f', 'BackEnd/app/Database/postgre_db.py': 'd17124c6ac8a37fb34b5e96acee18dca931772af7d59519087ea340f23e2f1f0', 'BackEnd/app/Database/qdrant_db.py': 'b041716b17efbecef2cef9a9519f90c1c9cf6a62c01cb13440b860649e490aca', 'BackEnd/app/Retrieval/__init__.py': '72de3c5e075bf2e5b77ca13b9b89ae9b408fd66ad1edf5a8607c0f0b07c2609c', 'BackEnd/app/Retrieval/qwen_reranker.py': 'df8be5caab3998856c6e13631233c9e511c17f5fbc24ee2b5152b7f0638fe9a1', 'BackEnd/app/Retrieval/export.py': '557d32907868d8960a3370264fc04e72757dd81825ff80f893db20d8f5c63705', 'BackEnd/app/Retrieval/store.py': '5132bcf6772fe0fac61f6ca30d784ee4f984937cdfd9dd8b6968694729676847', 'BackEnd/app/Retrieval/utils.py': '2a295393c53a400906ee3e2613fc647d2b00849b9e9c38e59bde7bede1ba3f4d', 'Scripts/Kaggle/retrieval_runtime.py': 'a93416ba3893a3df0e80664b1221130bf798baa9ac8c9e98488396c93949040b', 'Scripts/Kaggle/check_retrieval.py': '5d9cb09cca6faa582e5f5a2420c9cdd5ced5da5c63f5fb57ad95099b986d18a9'}
packed = base64.b64decode('...')
if hashlib.sha256(packed).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Embedded source bundle checksum mismatch")
CODE = TEMP / "code"
CODE.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(packed)) as bundle:
    bundle.extractall(CODE)
for name, expected in SOURCE_HASHES.items():
    if hashlib.sha256((CODE / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Embedded source checksum mismatch: {name}")
sys.path.insert(0, str(CODE))
import pyarrow.parquet as pq
import psutil
from BackEnd.app.pipeline import run_pipeline, PipelineError, _columns, _embedding_columns
from BackEnd.app.Database.qdrant_db import QdrantManager
from Scripts.Kaggle.retrieval_runtime import (
    find_input, build_source_index, JsonSourceStore, SourceQdrant,
    restore_qdrant, stop_qdrant, validate_submission, compare_query_traces,
)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True,
                    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(OUT / "pipeline.log")])
RUN_STARTED = time.monotonic()
REPORT = {"success": False, "runtime": "Kaggle CPU", "source_hashes": SOURCE_HASHES,
          "bundle_sha256": BUNDLE_SHA256, "settings": {
              "rerank": RERANK, "search_mode": SEARCH_MODE, "branch_limit": BRANCH_LIMIT,
              "top_k": TOP_K, "workers": WORKERS, "batch_size": BATCH_SIZE,
              "timeout": REQUEST_TIMEOUT, "server_query_timeout": max(1, int(REQUEST_TIMEOUT) - 10),
              "query_retries": QUERY_RETRIES,
              "limit": LIMIT, "start": START, "end": END}}
(OUT / "run_report.json").write_text(json.dumps(REPORT, indent=2))
print(f"CPU={os.cpu_count()}, RAM={psutil.virtual_memory().total / 1024**3:.1f} GiB")
print("Pipeline source SHA-256:", SOURCE_HASHES["BackEnd/app/pipeline.py"])
subprocess.run([sys.executable, str(CODE / "Scripts/Kaggle/check_retrieval.py")],
               cwd=CODE, env={**os.environ, "PYTHONPATH": str(CODE)}, check=True)
REPORT["adapter_tests"] = "passed on Kaggle"


In [ ]:
# Inputs và chỉ mục SQLite nhỏ, không sao chép toàn bộ văn bản corpus.
query_path = find_input("query_rewritten_final_embeddings_weighted.parquet", QUERY_PATH)
if SOURCE_PATH:
    source_path = find_input("merged.jsonl", SOURCE_PATH)
else:
    source_matches = sorted(Path("/kaggle/input").rglob("merged.jsonl")) + sorted(Path("/kaggle/input").rglob("merged.json"))
    if len(source_matches) != 1:
        raise ValueError(f"Set SOURCE_PATH explicitly: {source_matches}")
    source_path = source_matches[0]
manifest_path = find_input("snapshot_manifest.json", SNAPSHOT_MANIFEST_PATH)
manifest = json.loads(manifest_path.read_text())
parquet = pq.ParquetFile(query_path)
mapping = _columns(parquet.schema_arrow, None)
_embedding_columns(parquet.schema_arrow)
os.environ["QDRANT_COLLECTION"] = manifest["collection"]
os.environ["QDRANT_URL"] = "http://127.0.0.1:6333"
os.environ.pop("DATABASE_URL", None)
print("Query:", query_path, "rows:", parquet.metadata.num_rows)
print("Corpus:", source_path, "Snapshot manifest:", manifest_path)
index_path = TEMP / "source.sqlite"
local_source = TEMP / ("corpus" + source_path.suffix)
REPORT["source_index"] = build_source_index(source_path, index_path, manifest, local_source=local_source)
store = JsonSourceStore(local_source, index_path)
atexit.register(store.close)
print(json.dumps(REPORT["source_index"], indent=2))


In [ ]:
# Restore bằng binary chính thức, đúng phiên bản/checksum; tất cả trạng thái trong temp.
process, qdrant_log, REPORT["restore"] = restore_qdrant(
    manifest_path, TEMP / "qdrant", search_threads=QDRANT_SEARCH_THREADS,
    timeout=RESTORE_TIMEOUT,
)
atexit.register(stop_qdrant, process, qdrant_log)
payload_trace, live_trace = [], []

def source_factory(encoder):
    return SourceQdrant.from_manifest(manifest_path, url=os.environ["QDRANT_URL"],
                                      dense_encoder=encoder, timeout=REQUEST_TIMEOUT)

def payload_factory(encoder):
    return SourceQdrant.from_manifest(manifest_path, url=os.environ["QDRANT_URL"],
                                      dense_encoder=encoder, timeout=REQUEST_TIMEOUT, ids_only=False,
                                      record_queries=payload_trace)

def live_check_factory(encoder):
    return SourceQdrant.from_manifest(manifest_path, url=os.environ["QDRANT_URL"],
                                      dense_encoder=encoder, timeout=REQUEST_TIMEOUT,
                                      record_queries=live_trace)

options = dict(
    manifest_path=manifest_path, qdrant_url=os.environ["QDRANT_URL"],
    search_mode=SEARCH_MODE, top_k=TOP_K, branch_limit=BRANCH_LIMIT,
    batch_size=BATCH_SIZE, workers=WORKERS, timeout=REQUEST_TIMEOUT,
    max_docs=MAX_DOCS, max_chunks=MAX_CHUNKS, query_filter=QUERY_FILTER,
    start=START, end=END, postgres_manager=store, device="cpu",
    resume_tag="kaggle-json-pointer-v1:" + BUNDLE_SHA256 + ":" + manifest["signature"]["source_sha256"],
)
if PARITY_QUERIES < 1:
    raise ValueError("PARITY_QUERIES phải >= 1 để kiểm tra đối chiếu trước khi chạy")
parity_limit = min(PARITY_QUERIES, LIMIT) if LIMIT is not None else PARITY_QUERIES
checks = OUT / "checks" / OUTPUT_NAME
checks.mkdir(parents=True, exist_ok=True)
(OUT / "run_report.json").write_text(json.dumps(REPORT, indent=2))
check_options = {**options, "workers": 1}

def parity_run(name, factory, trace=None):
    path = checks / (name + ".json")
    trace_path = checks / (name + ".queries.json")
    if path.exists():
        if trace is not None:
            trace.extend(json.loads(trace_path.read_text()))
        return json.loads(path.read_text())
    try:
        return run_pipeline(query_path, path, qdrant_factory=factory, limit=parity_limit,
                            resume=RESUME and path.with_suffix(".checkpoint.jsonl").exists(), **check_options)
    except Exception as error:
        REPORT["parity_error"] = {"error_type": type(error).__name__,
                                  "failures": getattr(error, "failures", [])}
        (OUT / "run_report.json").write_text(json.dumps(REPORT, indent=2))
        print("Parity failure:", REPORT["parity_error"], flush=True)
        raise
    finally:
        if trace is not None:
            trace_path.write_text(json.dumps(trace, ensure_ascii=False))
        shutil.copyfile(TEMP / "qdrant" / "qdrant.log", OUT / "qdrant.log")

baseline = parity_run("payload", payload_factory, payload_trace)
live_hydrated = parity_run("json_live", live_check_factory, live_trace)
boundary_trace = []
control_manager = source_factory(None)
for recorded in payload_trace:
    body = {**recorded["body"], "limit": sum(p["limit"] for p in recorded["body"]["prefetch"])}
    result = control_manager._request("POST", control_manager._path + "/points/query", body)
    boundary_trace.append({"body": body, "result": result})
(checks / "cutoff_control.queries.json").write_text(json.dumps(boundary_trace, ensure_ascii=False))
live_comparison = compare_query_traces(payload_trace, live_trace, boundary_trace)
# Separate live RRF requests can permute tied hits. Replay identical ordered hits
# to verify that replacing PostgreSQL with JSON hydration preserves output exactly.
replay_trace = deepcopy(payload_trace)

def replay_factory(encoder):
    return SourceQdrant.from_manifest(manifest_path, url=os.environ["QDRANT_URL"],
                                      dense_encoder=encoder, timeout=REQUEST_TIMEOUT,
                                      replay_queries=replay_trace)

replayed = parity_run("json_replay", replay_factory)
if baseline != replayed:
    raise ValueError("JSON hydration changed the output for identical ordered Qdrant hits")
if not baseline or any(not p["relevant_chunks"] for p in baseline):
    raise ValueError("Đối chiếu không có ứng viên; kiểm tra snapshot/query trước khi chạy")
REPORT["parity"] = {"queries": len(baseline), "exact_predictions_match": True,
                    "baseline": "unchanged run_pipeline + original Qdrant payload",
                    "adapter": "unchanged run_pipeline + identical ordered hits + SQLite/JSON hydration",
                    "live_query_comparison": live_comparison}
print("PASS:", REPORT["parity"])
(OUT / "run_report.json").write_text(json.dumps(REPORT, indent=2))
del baseline, live_hydrated, replayed, payload_trace, live_trace, replay_trace, boundary_trace


In [ ]:
# Chạy toàn bộ query, checkpoint sau từng query, Qwen chỉ tải khi bật RERANK.
if RERANK:
    if importlib.util.find_spec("torch") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "torch>=2.6"], check=True)
    if importlib.util.find_spec("transformers") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
                        "transformers>=4.56,<6", "huggingface-hub>=0.30"], check=True)
    from huggingface_hub import snapshot_download
    from BackEnd.app.Retrieval.qwen_reranker import MODEL_NAME, MODEL_REVISION
    snapshot_download(MODEL_NAME, revision=MODEL_REVISION, cache_dir=TEMP / "huggingface" / "hub")
output = OUT / (OUTPUT_NAME + ".json")
archive = OUT / (OUTPUT_NAME + ".zip")
checkpoint = output.with_suffix(".checkpoint.jsonl")

def progress(state):
    if state["completed"] % BATCH_SIZE == 0 or state["completed"] == state["total"]:
        print(f"Progress {state['completed']}/{state['total']}; failures={state['failed']}", flush=True)

try:
    if not output.exists():
        for attempt in range(QUERY_RETRIES + 1):
            try:
                predictions = run_pipeline(
                    query_path, output, zip_path=archive, checkpoint_path=checkpoint,
                    resume=RESUME and checkpoint.exists(), limit=LIMIT, qdrant_factory=source_factory,
                    rerank=RERANK, reranker_device="cpu", reranker_batch_size=RERANKER_BATCH_SIZE,
                    reranker_max_length=RERANKER_MAX_LENGTH, reranker_token_budget=RERANKER_TOKEN_BUDGET,
                    model_cache=TEMP / "huggingface" / "hub", progress_callback=progress, **options,
                )
                break
            except PipelineError as error:
                transient = {"ConnectionError", "ConnectTimeout", "ReadTimeout", "Timeout", "ChunkedEncodingError"}
                if not RESUME or attempt == QUERY_RETRIES or any(
                    f["stage"] != "retrieval" or f["error_type"] not in transient for f in error.failures
                ):
                    raise
                REPORT.setdefault("retrieval_retries", []).append({"attempt": attempt + 1, "failures": error.failures})
                (OUT / "run_report.json").write_text(json.dumps(REPORT, indent=2))
                print("Retrying transient retrieval failures from checkpoint:", error.failures, flush=True)
                time.sleep(2)
        del predictions
    else:
        print("Đã có submission; kiểm tra lại ở cell tiếp theo.")
except BaseException as error:
    REPORT["error_type"] = type(error).__name__
    REPORT["seconds"] = time.monotonic() - RUN_STARTED
    (OUT / "run_report.json").write_text(json.dumps(REPORT, indent=2))
    shutil.copyfile(TEMP / "qdrant" / "qdrant.log", OUT / "qdrant.log")
    raise


In [ ]:
# Kiểm tra toàn bộ schema/ID/thứ tự, từng chunk nguyên văn và nội dung ZIP.
first = (START or 1) - 1
stop = END if END is not None else parquet.metadata.num_rows
if LIMIT is not None:
    stop = min(stop, first + LIMIT)
expected_ids = pq.read_table(query_path, columns=[mapping["id"]]).column(0).to_pylist()[first:stop]
REPORT["validation"] = validate_submission(output, archive, checkpoint, expected_ids, store, start_row=first)
REPORT["seconds"] = time.monotonic() - RUN_STARTED
REPORT["resources"] = {
    "python_rss_gib": psutil.Process().memory_info().rss / 1024**3,
    "qdrant_rss_gib": psutil.Process(process.pid).memory_info().rss / 1024**3,
    "temp_free_gib": shutil.disk_usage(TEMP).free / 1024**3,
}
REPORT["success"] = True
(OUT / "run_report.json").write_text(json.dumps(REPORT, indent=2))
shutil.copyfile(TEMP / "qdrant" / "qdrant.log", OUT / "qdrant.log")
print(json.dumps(REPORT, indent=2))
print("Kết quả:", output, archive)
stop_qdrant(process, qdrant_log)
